# Payer Policy Intelligence Pipeline

Extract 12 PA parameters + Access Score (0–100) from PA policy PDFs.

**How this works**: PDFs are parsed to text, the brand-specific section is isolated, three grouped Llama-3.3-70B (Groq) prompts return structured JSON (scalars, step-graph, text fields), Python deterministically counts steps and computes the Access Score using a transparent rubric, and HTML audit cards make every decision auditable.

**To reproduce**: install requirements, set `GROQ_API_KEY`, run all cells. The pipeline caches every LLM call so re-runs are free.

## 1. Environment setup

In [ ]:
# In Colab / Kaggle:
# !pip install -q -r requirements.txt
# !apt-get install -y poppler-utils      # for pdftotext

import os
# os.environ['GROQ_API_KEY'] = 'your-key-here'  # set before running

import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from src import (
    config, ingest, extract_text, segment_brand, llm_client,
    extract_params, step_graph, validate, access_score,
    pipeline, evidence_report, visualize, holdout,
)
print('Modules loaded. Working dir:', config.PROJECT_ROOT)

## 2. Smoke test (1 row, ~10s with primed cache)

In [ ]:
rows = ingest.load_submissions()
print(f'Total target rows: {len(rows)}')
print(f'Total PDFs:        {len(ingest.list_pdfs())}')
diag = pipeline.process_row(rows[0], verbose=True)
if 'csv_row' in diag:
    print('\nFirst row output:')
    for k, v in diag['csv_row'].items():
        print(f'  {k}: {str(v)[:120]}')

## 3. Extract text for all 70 PDFs (cached)

In [ ]:
paths = extract_text.extract_all()
print(f'Cached {len(paths)} PDF text extractions in {config.TEXT_CACHE}')

## 4. Full 79-row extraction (writes result.csv)

In [ ]:
df = pipeline.run_all(run_self_consistency=False, verbose=True)
df.head()

## 5. Per-row audit cards + index

In [ ]:
paths = evidence_report.render_all()
evidence_report.render_index()
print(f'Rendered {len(paths)} audit cards in {config.AUDIT_DIR}')
print(f'Open {config.AUDIT_DIR / "index.html"} in a browser.')

## 6. Visualisations: restrictiveness heatmap + score distribution

In [ ]:
heatmap_path = visualize.render_heatmap()
dist_path = visualize.render_score_distribution()
from IPython.display import Image, display
display(Image(heatmap_path))
display(Image(dist_path))

## 7. Hold-out accuracy (after manually filling holdout/holdout_labels.csv)

In [ ]:
import pandas as pd
holdout_df = pd.read_csv(config.HOLDOUT_CSV)
has_labels = holdout_df.drop(columns=['Filename', 'Brand', 'layout_hint']).notna().any(axis=1).any()
if has_labels:
    precision, joined = holdout.evaluate()
    holdout.print_report(precision)
else:
    print('Holdout labels not yet filled. Template at:', config.HOLDOUT_CSV)